# 🤖 NLP Assignment 3 — Chatbot using Hugging Face Transformers

**Intern ID:** IN226066402  
**Name:** Sannidhya  
**Assignment:** Build a Chatbot using Hugging Face Transformers  

---

## 📌 Objective
Build a simple conversational chatbot using a pre-trained transformer model (DialoGPT) from Hugging Face that can interact with users and generate meaningful responses.

## 🛠️ Tools Used
- Python
- Hugging Face Transformers
- PyTorch
- DialoGPT (Microsoft's conversational AI model)

---

## Step 1 — Install Required Libraries

In [ ]:
# Install Hugging Face Transformers and PyTorch
# Run this cell only once
!pip install transformers torch --quiet
print("✅ Installation complete!")

## Step 2 — Import Libraries

In [ ]:
# Import necessary libraries
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

## Step 3 — Load Pre-trained Model and Tokenizer

We are using **DialoGPT-small** — a pre-trained conversational AI model by Microsoft, available on Hugging Face Model Hub.

- **DialoGPT** is fine-tuned on Reddit conversations
- It generates natural, conversational responses
- The tokenizer converts text to tokens the model can understand

In [ ]:
# Load the pre-trained DialoGPT model and tokenizer from Hugging Face

print("⏳ Loading DialoGPT model... (this may take a minute on first run)")
sys.stdout.flush()

MODEL_NAME = "microsoft/DialoGPT-small"

# Load tokenizer — converts text to token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')

# Load the causal language model for text generation
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode
model.eval()

print("✅ Model loaded successfully!")
print(f"Model: {MODEL_NAME}")

## Step 4 — Define Response Generation Function

This function:
1. Takes user input and conversation history
2. Encodes the input using the tokenizer
3. Generates a response using the model
4. Decodes the response back to text

In [ ]:
def generate_response(user_input, chat_history_ids=None):
    """
    Generate a chatbot response using DialoGPT.

    Parameters:
        user_input (str): The message typed by the user
        chat_history_ids (Tensor or None): Previous conversation context

    Returns:
        response (str): The chatbot's reply
        chat_history_ids (Tensor): Updated conversation history
    """

    # Encode user input and add end-of-string token
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    # Append new input to conversation history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Keep conversation history to max 1000 tokens to avoid memory issues
    if bot_input_ids.shape[-1] > 1000:
        bot_input_ids = bot_input_ids[:, -1000:]

    # Build attention mask
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # Generate response using the model
    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_new_tokens=150,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.75,
        )

    # Decode only the new response (not the entire history)
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

print("✅ Response generation function defined!")

## Step 5 — Test Single Response

Let's test the model with a single input before running the full chatbot.

In [ ]:
# Test the model with a single message
test_input = "Hello! How are you?"

print("⏳ Generating response...")
sys.stdout.flush()

test_response, _ = generate_response(test_input)

print(f"User: {test_input}")
print(f"Chatbot: {test_response}")
print("\n✅ Model is working correctly!")

## Step 6 — Build the Interactive Chatbot

The chatbot:
- Greets the user when started
- Accepts user input in a loop
- Generates and displays responses
- Maintains conversation history for context
- Exits when user types `exit` or `quit`

> 💡 After running this cell, type your message in the input box that appears below and press **Enter**.

In [ ]:
def run_chatbot():
    """
    Run the interactive chatbot.
    Maintains conversation history for multi-turn dialogue.
    Type 'exit' or 'quit' to stop.
    """

    print("=" * 60)
    print("🤖 Welcome to AI Chatbot powered by DialoGPT")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 60)
    print("(Type 'exit' or 'quit' to end the conversation)")
    print("-" * 60)
    sys.stdout.flush()

    chat_history_ids = None

    while True:

        user_input = input("\nYou: ").strip()

        if not user_input:
            print("Chatbot: Please type something!")
            sys.stdout.flush()
            continue

        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: Thank you for chatting with me! Goodbye! 👋")
            print("=" * 60)
            sys.stdout.flush()
            break

        print("⏳ Thinking...")
        sys.stdout.flush()

        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        print(f"Chatbot: {response}")
        sys.stdout.flush()

# Run the chatbot
run_chatbot()

## Step 7 — Sample Conversation Output

Below is a sample conversation demonstrating the chatbot's capabilities:

In [ ]:
# Simulate a sample conversation to demonstrate chatbot output

sample_conversations = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "Tell me something interesting",
    "Thank you"
]

print("=" * 60)
print("📝 Sample Conversation Demonstration")
print("=" * 60)
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
sys.stdout.flush()

demo_history = None

for user_msg in sample_conversations:
    print(f"\nUser: {user_msg}")
    print("⏳ Thinking...")
    sys.stdout.flush()
    response, demo_history = generate_response(user_msg, demo_history)
    print(f"Chatbot: {response}")
    sys.stdout.flush()

print("\nUser: exit")
print("Chatbot: Thank you for chatting with me! Goodbye! 👋")
print("=" * 60)

## 📊 Summary

| Component | Details |
|-----------|----------|
| **Model Used** | microsoft/DialoGPT-small |
| **Framework** | Hugging Face Transformers + PyTorch |
| **Input Handling** | Console-based user input with loop |
| **Response Generation** | Prompt-based text generation with sampling |
| **Conversation Memory** | Maintained via chat_history_ids tensor |
| **Exit Condition** | 'exit' or 'quit' keywords |

## 🔑 Key Concepts Learned

1. **Transformer Models** — DialoGPT is a GPT-2 based model fine-tuned on conversational data
2. **Tokenization** — Converting text to numerical tokens the model understands
3. **Text Generation** — Using `model.generate()` with sampling parameters
4. **Conversation History** — Appending previous context to maintain dialogue flow
5. **Hugging Face Hub** — Loading pre-trained models with `AutoModelForCausalLM`

---
*Built for Data Science Internship — February 2026 · NLP Assignment 3 🚀*